In [ ]:
from parcels import (
    FieldSet,
    JITParticle,
    ScipyParticle,
    ParticleSet,
    AdvectionRK4,
    AdvectionRK4_3D,
    StatusCode,
)
import parcels

import datetime as dt
from datetime import datetime, timedelta

import dask
from dask.distributed import Client

import numpy as np

from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cmocean

import xarray as xr
from glob import glob
import cartopy
import cartopy.crs as ccrs
import xoak
from time import time
import warnings

warnings.simplefilter("ignore")

# Set Parameters
- [x] aussetzTiefe = min seasurface hight um zu verhindern dass Partikel in die Luft gesetzt werden
- [x] Speicherkoog & Wilhelmshaven vor Watt setzen
- [x] Sicherstellen, dass es keine 1d aussetzregionen gibt
- [ ] Aussetzen mit dichte verteilung (Volumen Errechnen mit median/mean Tiefe und darein 1000 partikel) 
- [x] Was ist bei Site 7 Flasch gelaufen?

In [ ]:
# Parameters
RNG_seed = 123

save_path = "/gxfs_work/geomar/smomw597/2025_copepods/output/Trajectories/"

# Time
year = 2021
start_month = 6
end_month = 11
start_day = 1
end_day = 1
max_age_d = 28
# timedirection
timearrow = 1
# Timestep in minutes
dt_in_minutes = 15
output_dt_in_minutes = 15


# Box traits
min_depth_m = 0
max_depth_m = 25
box_side_length_km = 5

site_counter = 8

number_particles = 100

repeated_release = True
repeatdt_d = 1

isPapermill = False

In [ ]:
if not isPapermill:
    year = 2021
    start_month = 6
    end_month = 6
    start_day = 1
    end_day = 2
    max_age_d = 10
    # timedirection
    timearrow = 1
    # Timestep in minutes
    dt_in_minutes = 15
    output_dt_in_minutes = 15

    # Box traits
    min_depth_m = 0
    max_depth_m = 25
    box_side_length_km = 5

    site_counter = 5

    number_particles = 1000

    repeated_release = False
    repeatdt_d = 5

# Read Files

In [ ]:
# Get Variables from Parameters
start_date = np.datetime64(f"{year}-{start_month:02d}-{start_day:02d}", "D")
end_date = np.datetime64(f"{year}-{end_month:02d}-{end_day:02d}", "D")
first_day_in_year = np.datetime64(f"{year}-01-01", "D")

start_date_str = start_date.astype(str).replace("-","")
end_date_str = end_date.astype(str).replace("-","")

start_file = (start_date - first_day_in_year).astype(int) * 4-1
end_file = (end_date - first_day_in_year + 1).astype(int) * 4+1

runtime_in_days = (end_date - start_date).tolist()
dt_min = np.timedelta64(dt_in_minutes, "m").tolist()
dt_out_min = np.timedelta64(output_dt_in_minutes, "m").tolist()

np.random.seed(RNG_seed)
print(start_file,end_file)

In [ ]:
# Read Files
data_path_orig_files = Path("/gxfs_work/geomar/smomw400/bsh_operationalmodel_data/")
data_path_divz_files = Path("/gxfs_work/geomar/smomw122/bsh_operationalmodel_data/")
data_path_static_files = Path("/gxfs_work/geomar/smomw122/bsh_operationalmodel_data")
data_path_static_fine = data_path_static_files / "static_file_fine"
data_path_static_coarse = data_path_static_files / "static_file_coarse"

sigma_file_fine = data_path_static_fine / "sigma_file_fine.nc"
H0_file_fine = data_path_static_fine / "H0_file_fine.nc"
divH0_file_fine = data_path_static_fine / "divH0_file_fine.nc"
lonlat_file_fine = data_path_static_fine / "lonlat_file_fine.nc"

c_files_fine = sorted(data_path_orig_files.glob(f"c_file_fine_{year}/*"))[start_file:end_file]
z_files_fine = sorted(data_path_orig_files.glob(f"z_file_fine_{year}/*"))[start_file:end_file]
t_files_fine = sorted(data_path_orig_files.glob(f"t_file_fine_{year}/*"))[start_file:end_file]
divz_files_fine = sorted(data_path_divz_files.glob(f"divz_file_fine_{year}/*"))[start_file:end_file]

sigma_file_coarse = data_path_static_coarse / "sigma_file_coarse.nc"
H0_file_coarse = data_path_static_coarse / "H0_file_coarse.nc"
divH0_file_coarse = data_path_static_coarse / "divH0_file_coarse.nc"
lonlat_file_coarse = data_path_static_coarse / "lonlat_file_coarse.nc"

c_files_coarse = sorted(data_path_orig_files.glob(f"c_file_coarse_{year}/*"))[start_file:end_file]
z_files_coarse = sorted(data_path_orig_files.glob(f"z_file_coarse_{year}/*"))[start_file:end_file]
t_files_coarse = sorted(data_path_orig_files.glob(f"t_file_coarse_{year}/*"))[start_file:end_file]
divz_files_coarse = sorted(data_path_divz_files.glob(f"divz_file_coarse_{year}/*"))[start_file:end_file]

In [ ]:
# open eta and H0 files
ds_eta_fine = xr.open_dataset(z_files_fine[0])
ds_H0_fine = xr.open_dataset(H0_file_fine)

ds_eta_coarse = xr.open_dataset(z_files_coarse[0])
ds_H0_coarse = xr.open_dataset(H0_file_coarse)

# Functions

In [ ]:
# https://github.com/Yichabod/natural_disaster_pred/blob/master/cropping_coordinates.py

earth_radius = 6271.0
degrees_to_radians = np.pi / 180.0
radians_to_degrees = 180.0 / np.pi


def change_in_latitude(kms):
    "Given a distance north, return the change in latitude."
    return (kms / earth_radius) * radians_to_degrees


def change_in_longitude(latitude, kms):
    "Given a latitude and a distance west, return the change in longitude."
    # Find the radius of a circle around the earth at given latitude.
    r = earth_radius * np.cos(np.multiply(latitude, degrees_to_radians))
    return (kms / r) * radians_to_degrees

In [ ]:
# Draw uniformly distributed random positions on the globe
def get_uniform_random_latlon_in(
    lat_min, lat_max,
    lon_min, lon_max,
    number_particles,
):
    lat = np.random.uniform(
        lat_min, lat_max,
        size=number_particles,
    )
    lon = np.random.uniform(
        lon_min, lon_max,
        size=number_particles,
    )
    return lat, lon

# Samples

In [ ]:
samples = [
    "Speicherkoog", "KB03", "SW08",
    "Wilhelmshaven", "GB96", "H21",
    "Boknis_Eck", "Gdynia", "Riga",
    "IU7_c", "LL17_c", "Finland",
    "BB23", "BB36", "Speicherkoog_Außen",
    "Wilhelmshaven_Außen",
]
lats = [
    54.0929, 54.692, 54.413,
    53.513, 56.54, 54.666,
    54.5169, 54.5829, 57.3911,
    59.489, 59.02, 59.77,
    55.1751, 54.575, 54.11,
    53.637,
]
lons = [
    8.9487, 10.1035, 11.617,
    8.149, 19.5809, 13.0224,
    10.034, 18.6042, 23.8588,
    21.2021, 21.0478, 23.2663,
    15.4352, 16.001, 8.5,
    8.15,
]
# lats = [54.0929, 54.692, 54.413, 53.513, 56.54, 54.666, 54.3101, 54.5329, 57.3911, 59.489, 59.02, 59.7766, 55.1751, 54.575]
# lons = [8.9487, 10.1035, 11.617,8.149,19.5809,13.0224,10.0201,18.5642,23.8588,21.2021,21.0478,23.2663,15.4352,16.001]
print(samples[site_counter])
lat_site = lats[site_counter]
lon_site = lons[site_counter]

ds_H0_fine.H0.plot()
ds_H0_coarse.H0.plot()
plt.scatter(lons, lats)
plt.scatter(lon_site, lat_site, c="r")
plt.show()

In [ ]:
# Make release box
lat_release_min = lat_site - change_in_latitude(box_side_length_km / 2)
lat_release_max = lat_site + change_in_latitude(box_side_length_km / 2)
lon_release_min = lon_site - change_in_longitude(lat_site, box_side_length_km / 2)
lon_release_max = lon_site + change_in_longitude(lat_site, box_side_length_km / 2)

# Make Box around release box, to ensure that depth values are existent while establishing particles
lat_box_min = lat_site - change_in_latitude(box_side_length_km * 2)
lat_box_max = lat_site + change_in_latitude(box_side_length_km * 2)
lon_box_min = lon_site - change_in_longitude(lat_site, box_side_length_km * 2)
lon_box_max = lon_site + change_in_longitude(lat_site, box_side_length_km * 2)

In [ ]:
# Build masks
fine_mask = (
    (ds_eta_fine.lat >= lat_box_min)
    & (ds_eta_fine.lat <= lat_box_max)
    & (ds_eta_fine.lon >= lon_box_min)
    & (ds_eta_fine.lon <= lon_box_max)
)
coarse_mask = (
    (ds_eta_coarse.lat >= lat_box_min)
    & (ds_eta_coarse.lat <= lat_box_max)
    & (ds_eta_coarse.lon >= lon_box_min)
    & (ds_eta_coarse.lon <= lon_box_max)
)

In [ ]:
# Check if area is in fine grid or not
if fine_mask.sum() == 0:
    region_mask = coarse_mask
    elev = ds_eta_coarse.elev.isel(time=0, drop=True)
    h0 = ds_H0_coarse.H0
else:
    region_mask = fine_mask
    elev = ds_eta_fine.elev.isel(time=0, drop=True)
    h0 = ds_H0_fine.H0

# only seed in water
elev_mask = ~ elev.isnull()

# avoid weird H0 < 0 locations
h0_mask = h0 > 0

In [ ]:
# Check where to place the particles horizontaly and the fraction of valid cells
seed_here = (
    (elev_mask & h0_mask)
    .where(region_mask, drop=True)
    .astype(bool)
)
fraction_valid_horizontal = seed_here.mean().data
if fraction_valid_horizontal > 0:
    number_horizontal = int(number_particles / fraction_valid_horizontal * 1.25)
    seasurface_height = (
        (elev + h0)
        .where((region_mask & h0_mask), drop=True)
    )
else:
    print("Es ist ein Fehler aufgetreten")

In [ ]:
# Generate lats and lons
# uniformly distributed everywhere
# uniform in lat / lon =/= uniform in m2
release_lats = np.random.uniform(lat_release_min, lat_release_max, size=number_horizontal)
release_lons = np.random.uniform(lon_release_min, lon_release_max, size=number_horizontal)

# check validity of positions
# note that this is _not_ per water volume but per sigma!
seedable = seed_here.sel(
    lon=xr.DataArray(release_lons, dims="particle"),
    lat=xr.DataArray(release_lats, dims="particle"),
    method="nearest",
).data

# remove invalide positions and cut to legth afterwards
release_lons = release_lons[seedable][:number_particles]
release_lats = release_lats[seedable][:number_particles]

In [ ]:
# Get maximal depth in Area
depth_in_release_area = h0.where(region_mask, drop=True)

# Set the maximal depth of bathymetry to the given maximal depth of Particles
depth_here = depth_in_release_area.where(
    depth_in_release_area < max_depth_m, 
    max_depth_m,
)

# Calculate the Sigma for each position
sigma_here = depth_here / seasurface_height

# Check the maximal sigma value for each particle
max_sigma_here = sigma_here.sel(
    lon=xr.DataArray(release_lons, dims="particle"),
    lat=xr.DataArray(release_lats, dims="particle"),
    method="nearest",
).data

# Check the elevation for each particle
elev_here = elev.where(
        region_mask & h0_mask, 
        drop=True,
    ).sel(
        lon=xr.DataArray(release_lons, dims="particle"),
        lat=xr.DataArray(release_lats, dims="particle"),
        method="nearest",
    ).data


# generate the depth for each particle between the maximal sigma and minimal elevation
depth = np.random.uniform(elev_here, max_sigma_here)

# Parcels

## Custom Kernel

In [ ]:
# Check, wether a Particle is above 25m depth and if not, move it to 24m
def FloatParticle(particle, fieldset, time):
    if particle.depth > max_depth_m:
        particle.depth = max_depth_m - 1

def DeleteErrorParticle(particle, fieldset, time):
    if particle.state >= 50:  # This captures all Errors
        particle.delete()

# Establish aging for particles
def Aging(particle, fieldset, time):
    particle.age_sec += particle.dt
    max_age_min = fieldset.max_age_d * 60 * 60 * 24
    if particle.age_sec > max_age_min:
        particle.delete()

In [ ]:
def AdvectionRK4_3D_SIGMABSH(particle, fieldset, time):
    # max_depth_m = 25
    max_depth_m = fieldset.max_depth
    time0 = time
    sig0 = particle.depth
    lat0 = particle.lat
    lon0 = particle.lon

    (u1, v1) = fieldset.UV[time0, sig0, lat0, lon0]  # horizontal velocities in deg/s

    w1 = fieldset.Wz[time0, sig0, lat0, lon0]  # this is upward in m/s rel to sig0 level

    s1 = fieldset.S[time0, sig0, lat0, lon0]
    t1 = fieldset.T[time0, sig0, lat0, lon0]

    eta1 = fieldset.eta[time0, 0, lat0, lon0]  # sea level elevation
    detadlon1 = fieldset.detadlon[time0, 0, lat0, lon0]
    detadlat1 = fieldset.detadlat[time0, 0, lat0, lon0]
    h01 = fieldset.H0[0, 0, lat0, lon0]  # reference bottom depth (for eta=0)
    dh0dlon1 = fieldset.dH0dlon[0, 0, lat0, lon0]
    dh0dlat1 = fieldset.dH0dlat[0, 0, lat0, lon0]
    h1 = h01 + eta1  # total height of water column

    wsigma1 = -w1 / h1 - sig0 / h1 * (
        u1 * (detadlon1 + dh0dlon1) + v1 * (detadlat1 + dh0dlat1)
    )

    time1 = time0 + 0.5 * particle.dt
    sig1 = max(
        0.0,
        min(min(1.0, max_depth_m / (eta1 + h01)), sig0 + wsigma1 * 0.5 * particle.dt),
    )
    # sig1 = max(0.0, min(1.0, sig0 + wsigma1 * 0.5 * particle.dt))

    lat1 = lat0 + v1 * 0.5 * particle.dt
    lon1 = lon0 + u1 * 0.5 * particle.dt

    (u2, v2) = fieldset.UV[time1, sig1, lat1, lon1]

    w2 = fieldset.Wz[time1, sig1, lat1, lon1]

    eta2 = fieldset.eta[time1, 0, lat1, lon1]
    detadlon2 = fieldset.detadlon[time1, 0, lat1, lon1]
    detadlat2 = fieldset.detadlat[time1, 0, lat1, lon1]
    h02 = fieldset.H0[0, 0, lat1, lon1]
    dh0dlon2 = fieldset.dH0dlon[0, 0, lat1, lon1]
    dh0dlat2 = fieldset.dH0dlat[0, 0, lat1, lon1]
    h2 = h02 + eta2

    wsigma2 = -w2 / h2 - sig1 / h2 * (
        u2 * (detadlon2 + dh0dlon2) + v2 * (detadlat2 + dh0dlat2)
    )

    time2 = time0 + 0.5 * particle.dt
    sig2 = max(
        0.0,
        min(min(1.0, max_depth_m / (eta2 + h02)), sig0 + wsigma2 * 0.5 * particle.dt),
    )
    # sig2 = max(0.0, min(1.0, sig0 + wsigma2 * 0.5 * particle.dt))
    lat2 = lat0 + v2 * 0.5 * particle.dt
    lon2 = lon0 + u2 * 0.5 * particle.dt

    (u3, v3) = fieldset.UV[time2, sig2, lat2, lon2]

    w3 = fieldset.Wz[time2, sig2, lat2, lon2]

    eta3 = fieldset.eta[time2, 0, lat2, lon2]
    detadlon3 = fieldset.detadlon[time2, 0, lat2, lon2]
    detadlat3 = fieldset.detadlat[time2, 0, lat2, lon2]
    h03 = fieldset.H0[0, 0, lat2, lon2]
    dh0dlon3 = fieldset.dH0dlon[0, 0, lat2, lon2]
    dh0dlat3 = fieldset.dH0dlat[0, 0, lat2, lon2]
    h3 = h03 + eta3

    wsigma3 = -w3 / h3 - sig2 / h3 * (
        u3 * (detadlon3 + dh0dlon3) + v3 * (detadlat3 + dh0dlat3)
    )

    time3 = time0 + particle.dt
    sig3 = max(
        0.0, min(min(1.0, max_depth_m / (eta3 + h03)), sig0 + wsigma3 * particle.dt)
    )
    # sig3 = max(0.0, min(1.0, sig0 + wsigma3 * particle.dt))
    lat3 = lat0 + v3 * particle.dt
    lon3 = lon0 + u3 * particle.dt

    (u4, v4) = fieldset.UV[time3, sig3, lat3, lon3]

    w4 = fieldset.Wz[time3, sig3, lat3, lon3]

    eta4 = fieldset.eta[time3, 0, lat3, lon3]
    detadlon4 = fieldset.detadlon[time3, 0, lat3, lon3]
    detadlat4 = fieldset.detadlat[time3, 0, lat3, lon3]
    h04 = fieldset.H0[0, 0, lat3, lon3]
    dh0dlon4 = fieldset.dH0dlon[0, 0, lat3, lon3]
    dh0dlat4 = fieldset.dH0dlat[0, 0, lat3, lon3]
    h4 = h04 + eta4

    wsigma4 = -w4 / h4 - sig3 / h4 * (
        u4 * (detadlon4 + dh0dlon4) + v4 * (detadlat4 + dh0dlat4)
    )

    lon4 = lon0 + (u1 + 2 * u2 + 2 * u3 + u4) / 6 * particle.dt
    lat4 = lat0 + (v1 + 2 * v2 + 2 * v3 + v4) / 6 * particle.dt
    sig4 = max(
        0.0,
        min(
            min(1.0, max_depth_m / (eta4 + h04)),
            sig0 + (wsigma1 + 2 * wsigma2 + 2 * wsigma3 + wsigma4) / 6 * particle.dt,
        ),
    )
    # sig4 = max(0.0, min(1.0, sig0 + (wsigma1 + 2 * wsigma2 + 2 * wsigma3 + wsigma4) / 6 * particle.dt))

    particle_dlon += lon4 - lon0
    particle_dlat += lat4 - lat0
    particle_ddepth += sig4 - sig0

    particle.eta = eta1
    particle.h0 = h01
    particle.wz = w1
    particle.u = u1
    particle.v = v1
    particle.wsigma = wsigma1
    particle.S = s1
    particle.T = t1
    particle.d = particle.depth

In [ ]:
CustomKernel = [AdvectionRK4_3D_SIGMABSH, Aging, DeleteErrorParticle]

## Particles

In [ ]:
# Establish partivle variables
particle_variables = (
    "eta", "h0", "wz", 
    "u", "v", "wsigma", 
    "S", "T", "d",
)
SampleParticle = parcels.JITParticle.add_variables(particle_variables)
SampleParticle = SampleParticle.add_variable("age_sec", initial=0)

In [ ]:
dim_dict_lonlat = {"lon": "lon", "lat": "lat"}
dim_dict_lonlat_time = dict(dim_dict_lonlat, time="time")
dim_dict_lonlat_time_depth = dict(dim_dict_lonlat_time, depth = "sigma")

In [ ]:
# Prepare reading of variables
fieldset_variables = [
    "U", "V", "Wz", 
    "H0", "dH0dlon", "dH0dlat",
    "eta", "detadlon", "detadlat",
]
variable_names = ["uvel", "vvel", "wvel", 
    "H0", "dH0dlon", "dH0dlat",
    "eta", "detadlon", "detadlat"
]
dim_dicts = [
    dim_dict_lonlat_time_depth, dim_dict_lonlat_time_depth, dim_dict_lonlat_time_depth,
    dim_dict_lonlat, dim_dict_lonlat, dim_dict_lonlat, 
    dim_dict_lonlat_time, dim_dict_lonlat_time, dim_dict_lonlat_time,
]
interp_methods = [
    "cgrid_velocity", "cgrid_velocity", "cgrid_velocity",
    "cgrid_tracer", "cgrid_tracer", "cgrid_tracer",
    "cgrid_velocity", "cgrid_velocity", "cgrid_velocity",
]

variables = dict(zip(fieldset_variables, variable_names))
dimensions = dict(zip(fieldset_variables, dim_dicts))
interp_method = dict(zip(fieldset_variables, interp_methods))

In [ ]:
# Prepare reading of variables
variables = {
    "U": "uvel", "V": "vvel", "Wz": "wvel", 
    "H0": "H0", "dH0dlon": "dH0dlon", "dH0dlat": "dH0dlat",
    "eta": "elev", "detadlon": "detadlon", "detadlat": "detadlat",
}

dimensions = {
    "U": dim_dict_lonlat_time_depth,
    "V": dim_dict_lonlat_time_depth,
    "Wz": dim_dict_lonlat_time_depth,
    "H0": dim_dict_lonlat,
    "dH0dlon": dim_dict_lonlat,
    "dH0dlat": dim_dict_lonlat,
    "eta": dim_dict_lonlat_time,
    "detadlon": dim_dict_lonlat_time,
    "detadlat": dim_dict_lonlat_time,
}

interp_method = {
    "U": "cgrid_velocity", "V": "cgrid_velocity", "Wz": "cgrid_velocity",
    "H0": "cgrid_tracer", "dH0dlon": "cgrid_tracer", "dH0dlat": "cgrid_tracer",
    "eta": "cgrid_velocity", "detadlon": "cgrid_velocity", "detadlat": "cgrid_velocity",
}

## Fieldset

### fine fieldset

In [ ]:
lonlat_dict_fine = {
    "lon": lonlat_file_fine,
    "lat": lonlat_file_fine,
}
st_dict_fine = dict(
    lonlat_dict_fine,
    depth=sigma_file_fine,
    data=t_files_fine,
)
current_dict_fine = dict(
    lonlat_dict_fine, 
    depth=sigma_file_fine,
    data=c_files_fine,
)
H0_dict_fine = dict(lonlat_dict_fine, data=H0_file_fine)
divH0_dict_fine = dict(lonlat_dict_fine, data=divH0_file_fine)
divz_dict_fine = dict(lonlat_dict_fine, data=divz_files_fine)
eta_dict_fine = dict(lonlat_dict_fine, data=z_files_fine)

In [ ]:
# Build fine fieldset for Temp and salinity
st_names_fine = {
    "S": st_dict_fine,
    "T": st_dict_fine,
}
st_fieldset_fine = FieldSet.from_netcdf(
    filenames=st_names_fine,
    variables={
        "S": "salt",
        "T": "temp",
    },
    dimensions={
        "S": dim_dict_lonlat_time_depth,
        "T": dim_dict_lonlat_time_depth,
    },
    interp_method={
        "S": "cgrid_tracer",
        "T": "cgrid_tracer",
    },
    allow_time_extrapolation=False,
    gridindexingtype="nemo",
)

In [ ]:
# Build fine fieldset for all the other Values
filenames_fine = {
    "V": current_dict_fine,
    "U": current_dict_fine,
    "Wz": current_dict_fine,
    "H0": H0_dict_fine,
    "dH0dlon": divH0_dict_fine,
    "dH0dlat": divH0_dict_fine,
    "eta": eta_dict_fine,
    "detadlon": divz_dict_fine,
    "detadlat": divz_dict_fine,
}

fieldset_fine = FieldSet.from_netcdf(
    filenames=filenames_fine,
    variables=variables,
    dimensions=dimensions,
    interp_method=interp_method,
    allow_time_extrapolation=False,
    gridindexingtype="nemo",
)

### coarse fieldsets

In [ ]:
lonlat_dict_coarse = {
    "lon": lonlat_file_coarse,
    "lat": lonlat_file_coarse,
}
st_dict_coarse = dict(
    lonlat_dict_coarse,
    depth=sigma_file_coarse,
    data=t_files_coarse,
)
current_dict_coarse = dict(
    lonlat_dict_coarse, 
    depth=sigma_file_coarse,
    data=c_files_coarse,
)
H0_dict_coarse = dict(lonlat_dict_coarse, data=H0_file_coarse)
divH0_dict_coarse = dict(lonlat_dict_coarse, data=divH0_file_coarse)
eta_dict_coarse = dict(lonlat_dict_coarse, data=z_files_coarse)
divz_dict_coarse = dict(lonlat_dict_coarse, data=divz_files_coarse)

In [ ]:
# Build coarse fieldset for Temp and salinity
st_names_coarse = {
    "S": st_dict_coarse,
    "T": st_dict_coarse,
}
st_fieldset_coarse = FieldSet.from_netcdf(
    filenames=st_names_coarse,
    variables={
        "S": "salt",
        "T": "temp",
    },
    dimensions={
        "S": dim_dict_lonlat_time_depth,
        "T": dim_dict_lonlat_time_depth,
    },
    interp_method={
        "S": "cgrid_tracer",
        "T": "cgrid_tracer",
    },
    allow_time_extrapolation=False,
    gridindexingtype="nemo",
)

In [ ]:
# Build coarse fieldset for all the other Values
filenames_coarse = {
    "U": current_dict_coarse,
    "V": current_dict_coarse,
    "Wz": current_dict_coarse,
    "H0": H0_dict_coarse,
    "dH0dlon": divH0_dict_coarse,
    "dH0dlat": divH0_dict_coarse,
    "eta": eta_dict_coarse,
    "detadlon": divz_dict_coarse,
    "detadlat": divz_dict_coarse,
}

fieldset_coarse = FieldSet.from_netcdf(
    filenames=filenames_coarse,
    variables=variables,
    dimensions=dimensions,
    interp_method=interp_method,
    allow_time_extrapolation=False,
    gridindexingtype="nemo",
)

### Nested fieldset

In [ ]:
# Build Nested field from the fine and coarse fields
U = parcels.NestedField("U", [fieldset_fine.U, fieldset_coarse.U])
V = parcels.NestedField("V", [fieldset_fine.V, fieldset_coarse.V])

nested_fieldset = FieldSet(U, V)

Wz_add = parcels.NestedField("Wz", [fieldset_fine.Wz, fieldset_coarse.Wz])
H0_add = parcels.NestedField("H0", [fieldset_fine.H0, fieldset_coarse.H0])
dH0dlon_add = parcels.NestedField("dH0dlon", [fieldset_fine.dH0dlon, fieldset_coarse.dH0dlon])
dH0dlat_add = parcels.NestedField("dH0dlat", [fieldset_fine.dH0dlat, fieldset_coarse.dH0dlat])
eta_add = parcels.NestedField("eta", [fieldset_fine.eta, fieldset_coarse.eta])
detadlon_add = parcels.NestedField("detadlon", [fieldset_fine.detadlon, fieldset_coarse.detadlon])
detadlat_add = parcels.NestedField("detadlat", [fieldset_fine.detadlat, fieldset_coarse.detadlat])
S_add = parcels.NestedField("S", [st_fieldset_fine.S, st_fieldset_coarse.S])
T_add = parcels.NestedField("T", [st_fieldset_fine.T, st_fieldset_coarse.T])

In [ ]:
# Combine the Nested Fields to a single nested field
nested_fieldset.add_field(Wz_add)
nested_fieldset.add_field(H0_add)
nested_fieldset.add_field(dH0dlon_add)
nested_fieldset.add_field(dH0dlat_add)
nested_fieldset.add_field(eta_add)
nested_fieldset.add_field(detadlon_add)
nested_fieldset.add_field(detadlat_add)
nested_fieldset.add_field(S_add)
nested_fieldset.add_field(T_add)
# Establish max depth and max age as constant in Fieldset
nested_fieldset.add_constant("max_depth", max_depth_m)
nested_fieldset.add_constant("max_age_d", max_age_d)

# Create Particles

In [ ]:
# Build Particle Set
start_time = np.datetime64(f'{year:04d}-{start_month:02d}-{start_day:02d}T00:00:00')

pset = ParticleSet(
    fieldset=nested_fieldset,
    pclass=SampleParticle,
    lat=release_lats,
    lon=release_lons,
    depth=depth,
    time=[start_time for n in range(number_particles)],
    repeatdt=timedelta(days=repeatdt_d),
)
if not repeated_release:
    pset.repeatdt = None

In [ ]:
filename_time = f"{start_date_str}-{end_date_str}_dt{output_dt_in_minutes}min"
filename_position = f"site{site_counter:02d}_d{min_depth_m}m-{max_depth_m}m"
filename = f"Nested_{filename_time}_{filename_position}_N{number_particles}_seed{RNG_seed}.zarr"
# define Output path and name
if isPapermill:
    output_filename = str("PPmill_" + filename)
else:
    output_filename = str("TEST_" + filename)

output_path = Path(save_path, output_filename)
print(f"{output_path}")

In [ ]:
# Define Outputparameters
output_particle_file = pset.ParticleFile(
    name=output_path,
    outputdt=dt_out_min,
    chunks=(number_particles, int(24 * 60 / output_dt_in_minutes)),
)

# Execute

In [ ]:
# Execute Simulation
pset.execute(
    CustomKernel,
    dt=dt_min,
    runtime=runtime_in_days,
    output_file=output_particle_file,
    verbose_progress=False,
)

# Analysis

## Read files

In [ ]:
# output_path = './output/TEST_Nested_20210601-20210602_dt50min_site8_d0m-25m_N10000_seed123.zarr'

In [ ]:
ds_trajectories = xr.open_zarr(output_path).compute()
ds_trajectories["age_day"] = (ds_trajectories.age_sec/(60*60*24)).compute()
depth_m = ds_trajectories.eta - ds_trajectories.z * (
    ds_trajectories.eta + ds_trajectories.h0
)
ds_trajectories = ds_trajectories.assign(depth=depth_m)
ds_trajectories

## Time plots

In [ ]:
# for i in np.arange(ds_trajectories.trajectory.shape[0] / number_particles):
#     traj_i = int(i*number_particles)
#     plt.plot(
#         ds_trajectories.time.isel(trajectory=traj_i),
#         ds_trajectories.age_sec.isel(trajectory=traj_i) / 3600,
#     )
# plt.show()

In [ ]:
# ds_trajectories.age_day.plot()

## Depth plots

In [ ]:
fig, ax = plt.subplots(
    1,2,
    figsize=(18,8),
)
ax[0].boxplot(depth_m.isel(obs=0))
ds_trajectories.isel(trajectory=slice(None, 100)).depth.plot(vmax=0, ax = ax[1])
plt.show()

In [ ]:
ds_trajectories.isel(trajectory=21).depth.plot()
plt.show()

## Maps

In [ ]:
# traj = ds_trajectories  # .isel(obs=671)#.isel(trajectory=6)
# # fieldset_NF
# # plt.pcolormesh(ds_eta_fine.elev.isel(time=0))
# ds_eta_fine.elev.isel(time=0).plot(
#     cmap=cmocean.cm.deep,
# )
# ds_eta_coarse.elev.isel(time=0).plot(
#     cmap=cmocean.cm.deep,
# )

# plt.scatter(
#     traj.lon,
#     traj.lat,
#     c="k",
#     s=0.00001,
# )
# plt.plot(
#     (
#         lon_release_max,
#         lon_release_max,
#         lon_release_min,
#         lon_release_min,
#         lon_release_max,
#     ),
#     (
#         lat_release_max,
#         lat_release_min,
#         lat_release_min,
#         lat_release_max,
#         lat_release_max,
#     ),
#     c="r",
# )
# # plt.xlim(7,27)
# # plt.ylim(53,60)
# plt.xlabel("Zonal distance [m]")
# plt.ylabel("Meridional distance [m]")
# plt.show()

In [ ]:
last_valid_obs = ds_trajectories.obs.where(ds_trajectories.lon.notnull()).max('obs').astype(int)
last_step = ds_trajectories.isel(obs=last_valid_obs).compute()

last_lon = last_step.lon
last_lat = last_step.lat

last_step.to_dataframe().describe()

In [ ]:
last_lon.values

In [ ]:
fig = plt.figure(figsize=(15,12))
ax = fig.add_subplot(
    1,1,1,
    projection= ccrs.PlateCarree(),
)
ax.add_feature(cartopy.feature.LAND, edgecolor='k')
ax.scatter(
    last_lon,
    last_lat,
    transform=ccrs.Geodetic(),
)
plt.show()

In [ ]:
fig, ax = plt.subplots(
    1,1,
    subplot_kw={'projection' : ccrs.PlateCarree()},
    figsize=(15,12),
)
ax.coastlines()
dense_plt = ax.hist2d(
    last_lon,
    last_lat,
    bins=[100, 100],
    norm=mcolors.LogNorm(),
    cmap=cmocean.cm.amp,
)
fig.colorbar(
    dense_plt[3],
    ax=ax,
    extend="max",
    label="density",
)
plt.show()